In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import warnings
import joblib
# from src.peptide_fw import *
from sklearn.linear_model import BayesianRidge
warnings.filterwarnings('ignore')

from paper_dataset_config import DATASET_SETTINGS_UPDATE, refid_all

SAVE_FIGS = True
SAVE_FIGS_DIR = 'results/paper/hi-res_figures'

In [ ]:
plt.rcParams.update({
    # 'font.family': 'serif',
    # 'font.serif': ['Times', 'Times', 'DejaVu Serif'],
    'font.size': 12,
    'font.weight': 'bold',
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 14,
    'axes.labelweight': 'normal',
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})
# how to auto-wrap long y-tick labels?
#A: matplotlib doesn't have built-in support for auto-wrapping y-tick labels, but we can manually insert line breaks into the labels based on a character limit. Here's a helper function to wrap text and how to apply it to y-tick labels:
def wrap_labels(labels, width):
    wrapped = []
    for label in labels:
        if len(label) > width:
            # Insert line breaks at space
            wrapped_label = ''
            current_line = ''
            for word in label.split():
                if len(current_line) + len(word) + 1 > width:
                    wrapped_label += current_line.rstrip() + '\n'
                    current_line = word + ' '
                else:
                    current_line += word + ' '
            wrapped_label += current_line.rstrip()
            wrapped.append(wrapped_label)
        else:
            wrapped.append(label)
    return wrapped

## Intergrate the results across series

In [ ]:
output_dir = './results/paper/'

# results_root_dir = './results/mdl_sel'
all_metrics = []  
# Loop over series
for target_name in ['Target_A', 'Target_B', 'Target_C', 'Target_D', 'Target_E']:
    settings = DATASET_SETTINGS_UPDATE[target_name]
    series_sel_all = settings['series_sel']
    for series_sel in series_sel_all:
        # load for ML models
        results_file = f"{output_dir}/performance_metrics_{target_name}_{series_sel}.csv"
        # results_file = f'{results_root_dir}/performance_metrics_{series}.csv'
        # Read the CSV file
        performance_metrics = pd.read_csv(results_file)
        all_metrics.append(performance_metrics)
# append chemeleon results
results_file = f"{output_dir}/performance_metrics_chemeleon_all.csv"
performance_metrics = pd.read_csv(results_file)
all_metrics.append(performance_metrics)
combined_metrics = pd.concat(all_metrics, ignore_index=True)

# # replace the pls performance
# results_file = f"{output_dir}/performance_metrics_pls_pcr.csv"
# combined_metrics = pd.concat([
#     combined_metrics[(combined_metrics['model'] != 'pls') & (combined_metrics['model'] != 'pcr')],
#     pd.read_csv(results_file)
# ], ignore_index=True)

# add the performance of linear svr
results_file = f"{output_dir}/performance_metrics_linear_svr.csv"
combined_metrics = pd.concat([
    combined_metrics,
    pd.read_csv(results_file)
], ignore_index=True)

combined_metrics['model_desc'] = combined_metrics.apply(lambda r: f"{r['model']}_{r['descriptor']}", axis=1)

linear_models = ['ridge', 'lasso', 'elastic_net', 'pls', 'pcr', 'bayesian_ridge'] # , 'linear_svr'
non_linear_mdls = ['rf', 'xgboost', 'svr']

# Define mapping dictionaries for display names
DESCRIPTOR_NAMES = {
    'mono_ecfp': 'ECFP4',
    'mono_rdk_fp': 'RDKitFP', 
    'mono_ohe': 'OHE',
    'mono_mordred': 'Mordred',
    'mono_rdk_desc': 'RDKitDesc',
    # 'pepland_orig_ckpt': 'Pepland original',
    # 'pepland_0130': 'Pepland',
}

MODEL_NAMES = {
    'bayesian_ridge': 'B-Ridge',
    'ridge': 'Ridge',
    'lasso': 'Lasso',
    'elastic_net': 'ElNet',
    # 'linear_svr': 'Linear SVR',
    'pls': 'PLS',
    'pcr': 'PCR',
    # 'mlp': 'MLP',
}

METRICS_NAMES = {
    'r2': '$R^{2}$',
    'spearman_corr': 'Spearman',
    'pearson_corr': 'Pearson'
}

SERIES_NAMES = {
    'Target_A_B-8154': 'A-1',
    'Target_A_B-8154 Macrolactam': 'A-2',
    'Target_A_B-8209': 'A-3',
    'Target_B_4R-5594': 'B-1',
    'Target_B_4R-2390': 'B-2',
    'Target_B_4R-9882': 'B-3',
    'Target_C_L-300_Macrolactam': 'C-1',
    'Target_C_L-320_Macrolactam': 'C-2',
    'Target_C_L-878': 'C-3',
    'Target_D_X-link 2-11': 'D-1',
    'Target_E_B-3503': 'E-1'
}

SPLIT_NAMES = {
    'cv': 'CV',
    'time': 'Time',
    'walkforward': 'Walk-forward'
}

print(combined_metrics.columns)
output_dir = './results/paper/'

# results_root_dir = './results/mdl_sel'
all_metrics = []  
# Loop over series
for target_name in ['Target_A', 'Target_B', 'Target_C', 'Target_D', 'Target_E']:
    settings = DATASET_SETTINGS_UPDATE[target_name]
    series_sel_all = settings['series_sel']
    for series_sel in series_sel_all:
        # load for ML models
        results_file = f"{output_dir}/performance_metrics_{target_name}_{series_sel}.csv"
        # results_file = f'{results_root_dir}/performance_metrics_{series}.csv'
        # Read the CSV file
        performance_metrics = pd.read_csv(results_file)
        all_metrics.append(performance_metrics)
# append chemeleon results
results_file = f"{output_dir}/performance_metrics_chemeleon_all.csv"
performance_metrics = pd.read_csv(results_file)
all_metrics.append(performance_metrics)
combined_metrics = pd.concat(all_metrics, ignore_index=True)

# # replace the pls performance
# results_file = f"{output_dir}/performance_metrics_pls_pcr.csv"
# combined_metrics = pd.concat([
#     combined_metrics[(combined_metrics['model'] != 'pls') & (combined_metrics['model'] != 'pcr')],
#     pd.read_csv(results_file)
# ], ignore_index=True)

# add the performance of linear svr
results_file = f"{output_dir}/performance_metrics_linear_svr.csv"
combined_metrics = pd.concat([
    combined_metrics,
    pd.read_csv(results_file)
], ignore_index=True)

combined_metrics['model_desc'] = combined_metrics.apply(lambda r: f"{r['model']}_{r['descriptor']}", axis=1)

linear_models = ['ridge', 'lasso', 'elastic_net', 'pls', 'pcr', 'bayesian_ridge'] # , 'linear_svr'
non_linear_mdls = ['rf', 'xgboost', 'svr']


# print(combined_metrics.columns)
# Index(['series', 'num_peptides', 'split_type', 'fold', 'descriptor', 'model',
#        'nrmse', 'r2', 'spearman_corr', 'pearson_corr', 'fine_tune_time',
#        'train_time', 'test_time', 'actual_params', 'model_desc'],
#       dtype='object')

In [ ]:
# fill the empty values in "spearman_corr", "pearson_corr" as 0:
combined_metrics['spearman_corr'] = combined_metrics['spearman_corr'].fillna(0)
combined_metrics['pearson_corr'] = combined_metrics['pearson_corr'].fillna(0)
combined_metrics.sample(5, random_state=42)

In [ ]:
pepland_results = combined_metrics[combined_metrics["descriptor"] == "pepland_0130"]
# pepland_results = pepland_results[pepland_results["model"] == "rf"]
pepland_results[pepland_results['r2'] < -100].shape

### Fig 1 - heatmap of the model performance


In [ ]:
# Plot func define. 
def create_performance_heatmap(data, split_type, metric, descriptors, models, figsize=(12, 8), colormap='viridis', show_std=True, show_rank=True, annotations_fontsize=12):
    """
    Create a 2D heatmap for model performance metrics with std and rank annotations
    
    Parameters:
    - data: DataFrame with performance metrics
    - split_type: 'random' or 'time'
    - metric: performance metric to visualize
    - figsize: figure size tuple
    - colormap: colormap to use ('jet', 'viridis', 'plasma', 'coolwarm', etc.)
    - show_std: whether to show standard deviation in annotations
    - show_rank: whether to show rank in annotations
    """
    
    # Filter data to exclude ridge_default
    filtered_data = data[(data['split_type'] == split_type) & 
                        (data['model'] != 'ridge_default')]
    
    # Group by descriptor and model, calculate mean and std
    grouped_stats = filtered_data.groupby(['descriptor', 'model'])[metric].agg(['mean', 'std']).reset_index()
    
    # Create pivot tables for mean, std
    pivot_mean = grouped_stats.pivot(index='descriptor', columns='model', values='mean')
    pivot_std = grouped_stats.pivot(index='descriptor', columns='model', values='std')
    
    # Calculate ranks (higher is better for r2, spearman_corr, pearson_corr; lower is better for nrmse)
    if metric == 'nrmse':
        # For NRMSE, lower is better (rank 1 = lowest value)
        ranks = pivot_mean.values.flatten()
        ranks = pd.Series(ranks).rank(method='min', ascending=True, na_option='bottom')
    else:
        # For R2 and correlations, higher is better (rank 1 = highest value)
        ranks = pivot_mean.values.flatten()
        ranks = pd.Series(ranks).rank(method='min', ascending=False, na_option='bottom')
    
    # Reshape ranks back to matrix form
    pivot_rank = pd.DataFrame(ranks.values.reshape(pivot_mean.shape), 
                             index=pivot_mean.index, 
                             columns=pivot_mean.columns)
    
    # Reorder rows and columns to match specified order
    pivot_mean = pivot_mean.reindex(index=descriptors, columns=models)
    pivot_std = pivot_std.reindex(index=descriptors, columns=models)
    pivot_rank = pivot_rank.reindex(index=descriptors, columns=models)
    
    # Create annotation matrix
    annotations = np.empty_like(pivot_mean.values, dtype=object)
    
    for i in range(pivot_mean.shape[0]):
        for j in range(pivot_mean.shape[1]):
            mean_val = pivot_mean.iloc[i, j]
            std_val = pivot_std.iloc[i, j]
            rank_val = pivot_rank.iloc[i, j]
            
            if pd.isna(mean_val):
                annotations[i, j] = 'N/A'
            else:
                # Build annotation string
                annotation_parts = [f'{mean_val:.3f}']
                
                if show_std and not pd.isna(std_val):
                    annotation_parts.append(f'±{std_val:.3f}')
                
                if show_rank and not pd.isna(rank_val):
                    annotation_parts.append(f'(rank {int(rank_val)})')
                
                annotations[i, j] = '\n'.join(annotation_parts)
    
    # Map the index and column names to display names using global mappings
    pivot_mean.index = pivot_mean.index.map(DESCRIPTOR_NAMES)
    pivot_mean.columns = pivot_mean.columns.map(MODEL_NAMES)
    
    # Create the heatmap
    plt.figure(figsize=figsize)
    
    # Use the specified colormap
    if metric == 'nrmse':
        # For NRMSE (lower is better), use reversed colormap
        cmap = colormap + '_r' if not colormap.endswith('_r') else colormap
    else:
        # For R2 and correlations (higher is better)
        cmap = colormap
    
    # Create heatmap with custom annotations
    ax = sns.heatmap(pivot_mean, 
                annot=annotations, 
                fmt='', 
                cmap=cmap, 
                center=None,
                square=True,
                linewidths=0.5,
                cbar_kws={},
                annot_kws={'fontsize': annotations_fontsize, 'ha': 'center', 'va': 'center'})
    
    # setup the colorbar
    cbar = ax.collections[0].colorbar
    """ cbar.set_label(METRICS_NAMES.get(metric, metric), 
                   fontweight='bold',     # Make bold
                   fontsize=12,           # Font size
                   rotation=270,          # Vertical text
                   labelpad=15) """
    # make the colorbar label appears on top
    cbar.ax.set_title(METRICS_NAMES.get(metric, metric), fontsize=18, weight='bold', pad=10)
    # Create title with information about what's shown
    metric_display = METRICS_NAMES.get(metric, metric.upper())
    title_parts = [f'{metric_display} Performance - {split_type.capitalize()} Split']
    if show_std and show_rank:
        title_parts.append('Values: Mean ± Std (Rank)')
    elif show_std:
        title_parts.append('Values: Mean ± Std')
    elif show_rank:
        title_parts.append('Values: Mean (Rank)')
    
    font_size = 20
    plt.title('\n'.join(title_parts), fontsize=16, fontweight='bold')
    plt.xlabel('Model Type', fontsize=font_size, fontweight='bold')
    plt.ylabel('Descriptor Type', fontsize=font_size, fontweight='bold')
    plt.xticks(rotation=0, ha='center', fontsize=font_size, fontweight='bold')
    plt.yticks(rotation=60, fontsize=font_size, fontweight='bold')
    plt.tight_layout()
    
    if SAVE_FIGS:
        fig_filename = f'performance_heatmap_{metric}_{split_type}.svg'
        plt.savefig(os.path.join(SAVE_FIGS_DIR, fig_filename), format='svg', dpi=300)
        print(f"Saved heatmap to {os.path.join(SAVE_FIGS_DIR, fig_filename)}")

    return {
        'mean_values': pivot_mean,
        'std_values': pivot_std, 
        'rank_values': pivot_rank
    }

def collect_results_stat_as_cols(data, split_type, metric, descriptors, models, show_std=True, show_rank=True,):
    """
    Collect performance metrics with std and rank as columns
    each row is one comb of descriptor -- model. 
    values are string type of "mean ± std (rank x)" (if args set to True)
    """
    # Filter data to exclude ridge_default
    filtered_data = data[(data['split_type'] == split_type) & 
                        (data['model'] != 'ridge_default')]
    
    # Group by descriptor and model, calculate mean and std
    grouped_stats = filtered_data.groupby(['descriptor', 'model'])[metric].agg(['mean', 'std']).reset_index()

    # Create pivot tables for mean, std
    pivot_mean = grouped_stats.pivot(index='descriptor', columns='model', values='mean')
    pivot_std = grouped_stats.pivot(index='descriptor', columns='model', values='std')
    # Calculate ranks (higher is better for r2, spearman_corr, pearson_corr; lower is better for nrmse)
    if metric == 'nrmse':
        # For NRMSE, lower is better (rank 1 = lowest value)
        ranks = pivot_mean.values.flatten()
        ranks = pd.Series(ranks).rank(method='min', ascending=True, na_option='bottom')
    else:
        # For R2 and correlations, higher is better (rank 1 = highest value)
        ranks = pivot_mean.values.flatten()
        ranks = pd.Series(ranks).rank(method='min', ascending=False, na_option='bottom')
    # Reshape ranks back to matrix form
    pivot_rank = pd.DataFrame(ranks.values.reshape(pivot_mean.shape), 
                             index=pivot_mean.index, 
                             columns=pivot_mean.columns)
    # Reorder rows and columns to match specified order
    pivot_mean = pivot_mean.reindex(index=descriptors, columns=models)
    pivot_std = pivot_std.reindex(index=descriptors, columns=models)
    pivot_rank = pivot_rank.reindex(index=descriptors, columns=models)

    # Create annotation matrix
    annotations = np.empty_like(pivot_mean.values, dtype=object)
    for i in range(pivot_mean.shape[0]):
        for j in range(pivot_mean.shape[1]):
            mean_val = pivot_mean.iloc[i, j]
            std_val = pivot_std.iloc[i, j]
            rank_val = pivot_rank.iloc[i, j]
            
            if pd.isna(mean_val):
                annotations[i, j] = 'N/A'
            else:
                # Build annotation string
                annotation_parts = [f'{mean_val:.3f}']
                
                if show_std and not pd.isna(std_val):
                    annotation_parts.append(f'±{std_val:.3f}')
                
                if show_rank and not pd.isna(rank_val):
                    annotation_parts.append(f'(rank {int(rank_val)})')
                
                annotations[i, j] = ' '.join(annotation_parts)

    # Convert annotations to DataFrame
                

    
    
    pass

In [ ]:
# Define descriptor and model orders
descriptors = list(DESCRIPTOR_NAMES.keys())
models = list(MODEL_NAMES.keys())
# extract the linear models
combined_metrics_linear = combined_metrics[(combined_metrics['model'].isin(linear_models))]

results_table_cols = []
results_table_col_names = []
for split_type in ['time', 'cv', 'walkforward']:
    for metric in ['r2', 'spearman_corr', 'pearson_corr']:
        result = create_performance_heatmap(combined_metrics_linear, split_type, metric, descriptors, models, show_std=True, show_rank=0, annotations_fontsize=16)

        results_table_cols.append(
            collect_results_stat_as_cols(
                combined_metrics_linear, split_type, metric, descriptors, models, show_std=True, show_rank=0, 
            )
        )

        results_table_col_names.append(f"")


### Fig 2 - Top configurations


In [ ]:
# extract and plot the top configurations by rank

'''
the columns in combined_metrics:
['series', 'num_peptides', 'split_type', 'descriptor', 'model', 'nrmse',
       'r2', 'spearman_corr', 'pearson_corr', 'fine_tune_time', 'train_time',
       'test_time', 'fold']
I want to calculate the averaged model performance for each descriptor-model combnation over each combination of split_type and performance metric including r2, spearman_corr, and pearson_corr.
Then, for each combination of split_type and performance metric, I want to rank the descriptor-model combnation. Here, I have 2 split types and three performanc metrics, so there will be six rank values for each descriptor-model combnation.
Then, I want to plot the top n descriptor-model combnations with the smallest avg. rank value. For each descriptor-model combnation, I want to plot its averaged performance metrics (six bars representing 2 split types and three performanc metrics). Also, I want to show the standard error of the mean (or std, make it an option) as errorbars of the bar plot. Also, I want the barplot to be horizontal in order to add text annotation for mean \pm std (rank X) for each bar. Here, the rank is the rank value. 

split_type = ['random', 'time']
descriptor = ['ecfp', 'rdk_fp','rdk_desc', 'mordred', 'ohe']
model = ['ridge', 'lasso', 'elastic_net', 'pls', 'pcr', 'bayesian_ridge', 'ridge_default']
'''

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


# Define colors for metrics (R², Spearman, Pearson)
METRIC_COLORS = ['#382E9B', '#C20202', '#00857C']

def analyze_model_performance(combined_metrics, n_top=10, error_type='sem', 
                            show_errorbar=True,
                            descriptor_names=None, model_names=None, 
                            metrics_names=None, metric_colors=None):
    """
    Analyze and visualize model performance across descriptor-model combinations.
    
    Parameters:
    -----------
    combined_metrics : pd.DataFrame
        DataFrame with model performance metrics
    n_top : int
        Number of top combinations to plot (default: 10)
    error_type : str
        Type of error bars: 'sem' for standard error of mean, 'std' for standard deviation
    show_errorbar : bool
        Whether to show error bars (default: True)
    descriptor_names : dict, optional
        Mapping of descriptor names to display names
    model_names : dict, optional
        Mapping of model names to display names
    metrics_names : dict, optional
        Mapping of metric names to display names
    metric_colors : list, optional
        List of colors for metrics [R², Spearman, Pearson]
    """
    
    # Use provided mappings or defaults
    descriptor_names = descriptor_names or DESCRIPTOR_NAMES
    model_names = model_names or MODEL_NAMES
    metrics_names = metrics_names or METRICS_NAMES
    metric_colors = metric_colors or METRIC_COLORS
    
    # Define the metrics and split types
    performance_metrics = ['r2', 'spearman_corr', 'pearson_corr']
    split_types = ['cv', 'time', 'walkforward']  # Reordered for better visualization
    
    # Create color mapping
    color_map = {metric: color for metric, color in zip(performance_metrics, metric_colors)}
    
    # Step 1: Calculate averaged performance for each descriptor-model combination
    print("Step 1: Calculating averaged performance metrics...")
    
    # Group by descriptor, model, and split_type, then calculate mean and std/sem
    grouped_stats = combined_metrics.groupby(['descriptor', 'model', 'split_type']).agg({
        'r2': ['mean', 'std', 'count'],
        'spearman_corr': ['mean', 'std', 'count'],
        'pearson_corr': ['mean', 'std', 'count']
    }).reset_index()
    
    # Flatten column names
    grouped_stats.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                           for col in grouped_stats.columns.values]
    
    # Calculate standard error of mean if requested
    if error_type == 'sem':
        for metric in performance_metrics:
            grouped_stats[f'{metric}_sem'] = grouped_stats[f'{metric}_std'] / np.sqrt(grouped_stats[f'{metric}_count'])
            
    print(grouped_stats)
    # Step 2: Rank descriptor-model combinations for each split_type and metric
    print("Step 2: Ranking combinations...")
    
    ranking_results = []
    
    for split_type in split_types:
        for metric in performance_metrics:
            # Filter data for current split_type
            split_data = grouped_stats[grouped_stats['split_type'] == split_type].copy()
            
            # Rank by metric (higher is better, so ascending=False)
            split_data[f'rank_{split_type}_{metric}'] = split_data[f'{metric}_mean'].rank(ascending=False)
            
            # Store ranking info
            ranking_info = split_data[['descriptor', 'model', f'rank_{split_type}_{metric}']].copy()
            ranking_info['split_type'] = split_type
            ranking_info['metric'] = metric
            ranking_info['rank'] = ranking_info[f'rank_{split_type}_{metric}']
            
            ranking_results.append(ranking_info[['descriptor', 'model', 'split_type', 'metric', 'rank']])
    
    # Combine all rankings
    all_rankings = pd.concat(ranking_results, ignore_index=True)
    
    # Step 3: Calculate average rank for each descriptor-model combination
    print("Step 3: Calculating average ranks...")
    
    avg_ranks = all_rankings.groupby(['descriptor', 'model'])['rank'].mean().reset_index()
    avg_ranks.columns = ['descriptor', 'model', 'avg_rank']
    avg_ranks = avg_ranks.sort_values('avg_rank')
    
    # Get top n combinations
    top_combinations = avg_ranks.head(n_top)
    
    print(f"\nTop {n_top} descriptor-model combinations by average rank:")
    for idx, row in top_combinations.iterrows():
        desc_display = descriptor_names.get(row['descriptor'], row['descriptor'])
        model_display = model_names.get(row['model'], row['model'])
        print(f"{desc_display} + {model_display}: avg rank = {row['avg_rank']:.2f}")
    
    # Step 4: Prepare data for plotting
    print("Step 4: Preparing visualization data...")
    
    plot_data = []
    
    for _, combo in top_combinations.iterrows():
        descriptor, model = combo['descriptor'], combo['model']
        
        for split_type in split_types:
            for metric in performance_metrics:
                # Get performance data
                mask = ((grouped_stats['descriptor'] == descriptor) & 
                       (grouped_stats['model'] == model) & 
                       (grouped_stats['split_type'] == split_type))
                
                if mask.sum() > 0:
                    row_data = grouped_stats[mask].iloc[0]
                    
                    # Get rank for this combination
                    rank_mask = ((all_rankings['descriptor'] == descriptor) & 
                               (all_rankings['model'] == model) & 
                               (all_rankings['split_type'] == split_type) & 
                               (all_rankings['metric'] == metric))
                    
                    rank_val = all_rankings[rank_mask]['rank'].iloc[0] if rank_mask.sum() > 0 else np.nan
                    
                    error_val = row_data[f'{metric}_sem'] if error_type == 'sem' else row_data[f'{metric}_std']
                    
                    plot_data.append({
                        'descriptor': descriptor,
                        'model': model,
                        'descriptor_model': f"{descriptor_names.get(descriptor, descriptor)} + {model_names.get(model, model)}",
                        'split_type': split_type,
                        'metric': metric,
                        'mean_performance': row_data[f'{metric}_mean'],
                        'error': error_val,
                        'rank': int(rank_val),
                        'avg_rank': combo['avg_rank']
                    })
    
    plot_df = pd.DataFrame(plot_data)
    
    # Step 5: Create the visualization
    print("Step 5: Creating visualization...")
    
    # Sort by average rank (best first) and reverse for plotting (best at top)
    plot_df = plot_df.sort_values('avg_rank')
    unique_combinations = plot_df['descriptor_model'].unique()
    
    # Create main figure
    fig, ax = plt.subplots(figsize=(12, max(8, n_top * 1.2)))
    
    # Calculate positions for bars - FIXED POSITIONING
    n_metrics = len(performance_metrics)
    n_splits = len(split_types)
    bar_height = 0.08  # Reduced bar height to prevent overlap
    combo_spacing = 1.0  # Increased spacing between combinations
    
    y_positions = []
    y_labels = []
    
    # Process combinations in reverse order to show best at top
    for i, combo in enumerate(reversed(unique_combinations)):
        combo_data = plot_df[plot_df['descriptor_model'] == combo]
        base_y = i * combo_spacing
        
        # Get average rank for this combination
        avg_rank = combo_data['avg_rank'].iloc[0]
        y_labels.append(f"{combo}\n(Avg Rank: {avg_rank:.1f})")
        y_positions.append(base_y)
        
        # FIXED: Create properly spaced bar positions
        bar_idx = 0
        for split_idx, split_type in enumerate(split_types):
            for metric_idx, metric in enumerate(performance_metrics):
                # Calculate position relative to base_y with proper spacing
                offset = (bar_idx - (n_metrics * n_splits - 1) / 2) * bar_height * 1.2
                bar_y = base_y + offset
                
                # Get data for this specific combination
                mask = ((combo_data['split_type'] == split_type) & 
                       (combo_data['metric'] == metric))
                split_data = combo_data[mask]
                
                if len(split_data) > 0:
                    row = split_data.iloc[0]
                    
                    # FIXED: Set distinct visual patterns for each split type
                    if split_type == 'cv':
                        hatch = None
                        alpha = 0.7
                    elif split_type == 'time':
                        hatch = '///'
                        alpha = 0.9
                    elif split_type == 'walkforward':
                        hatch = 'xxx'
                        alpha = 0.8
                    
                    # Create bar with or without error bars
                    if show_errorbar:
                        bar = ax.barh(bar_y, row['mean_performance'], 
                                     height=bar_height,
                                     xerr=row['error'], 
                                     color=color_map[metric], 
                                     alpha=alpha,
                                     hatch=hatch,
                                     capsize=2,
                                     error_kw={'capthick': 1})
                    else:
                        bar = ax.barh(bar_y, row['mean_performance'], 
                                     height=bar_height,
                                     color=color_map[metric], 
                                     alpha=alpha,
                                     hatch=hatch)
                    
                    # Add text annotation with performance value and rank
                    if show_errorbar:
                        text = f"{row['mean_performance']:.3f}±{row['error']:.3f} (rank {int(row['rank'])})"
                        text_x = row['mean_performance'] + row['error'] + 0.01
                    else:
                        text = f"{row['mean_performance']:.3f} (rank {int(row['rank'])})"
                        text_x = row['mean_performance'] + 0.01
                    
                    ax.text(text_x, bar_y, text, 
                           va='center', ha='left', fontsize=9)
                
                bar_idx += 1
    
    # Customize plot
    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)
    ax.tick_params(axis='y', labelsize=14)
    ax.tick_params(axis='x', labelsize=14)
    ax.set_xlabel('Performance Score', fontsize=14, fontweight='bold')
    
    # Update title based on whether error bars are shown
    if show_errorbar:
        title = f'Top {n_top} Descriptor-Model Combinations by Average Rank\nError bars show {error_type.upper()}'
    else:
        title = f'Top {n_top} Descriptor-Model Combinations by Average Rank'
    
    ax.set_title(title, fontsize=14, pad=20)
    
    # Adjust x-axis limits to accommodate text annotations
    x_min, x_max = ax.get_xlim()
    ax.set_xlim(x_min, x_max * 1.1)  # Increase right limit by 30%
    
    # Add grid for better readability
    ax.grid(axis='x', alpha=0.3)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.show()
    if SAVE_FIGS:
        fig_filename = f'model_performance_ranking_top_{n_top}.svg'
        fig.savefig(os.path.join(SAVE_FIGS_DIR, fig_filename), format='svg', dpi=300)
        print(f"Saved figure to {os.path.join(SAVE_FIGS_DIR, fig_filename)}")
    
    # Step 6: Create separate legend figure - FIXED LEGEND
    print("Step 6: Creating legend...")
    
    # Create legend figure
    legend_fig, legend_ax = plt.subplots(figsize=(10, 5))
    legend_ax.axis('off')  # Hide axes
    
    # FIXED: Create legend elements for all three split types
    legend_elements = []
    
    # Add walkforward split metrics (cross hatch)
    for metric in performance_metrics:
        legend_elements.append(plt.Rectangle((0,0),1,1, 
                                           facecolor=color_map[metric], 
                                           alpha=0.8,
                                           hatch='xxx',
                                           label=f"{metrics_names.get(metric, metric)} (Walk-forward)"))
    
    # Add time split metrics (diagonal hatch)
    for metric in performance_metrics:
        legend_elements.append(plt.Rectangle((0,0),1,1, 
                                           facecolor=color_map[metric], 
                                           alpha=0.9,
                                           hatch='///',
                                           label=f"{metrics_names.get(metric, metric)} (Time split)"))
        
    # Add CV split metrics (no hatch)
    for metric in performance_metrics:
        legend_elements.append(plt.Rectangle((0,0),1,1, 
                                           facecolor=color_map[metric], 
                                           alpha=0.7,
                                           label=f"{metrics_names.get(metric, metric)} (CV)"))
    
    
    
    # Create the legend with 3 columns for better organization
    legend = legend_ax.legend(handles=legend_elements, 
                             loc='center', 
                             ncol=3, 
                             fontsize=14,
                             # title='Performance Metrics by Split Type',
                             # title_fontsize=14
                             )
    
    plt.tight_layout()
    plt.show()

    if SAVE_FIGS:
        legend_fig_filename = f'model_performance_ranking_legend.svg'
        legend_fig.savefig(os.path.join(SAVE_FIGS_DIR, legend_fig_filename), format='svg', dpi=300)
        print(f"Saved legend to {os.path.join(SAVE_FIGS_DIR, legend_fig_filename)}")

    # Return summary statistics
    return {
        'top_combinations': top_combinations,
        'all_rankings': all_rankings,
        'grouped_stats': grouped_stats,
        'plot_data': plot_df
    }

# Example usage:
results = analyze_model_performance(combined_metrics_linear, n_top=5, error_type='sem', show_errorbar=False)


### Fig 3 - Comparison with non-linear models

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import f_oneway

# Selected methods
MODELDESC_KEEP = [
    'bayesian_ridge_mono_ecfp',
    'svr_mono_ecfp',
    'rf_mono_ecfp',
    'xgboost_mono_ecfp',
    'svr_pep_ecfp_counts',
    'rf_pep_ecfp_counts',
    'xgboost_pep_ecfp_counts',
    'chemeleon_graph',
    # 'mlp_pepland_0130'
]

# Exact display labels you provided
METHOD_DISPLAY = {
    'bayesian_ridge_mono_ecfp': 'PepOpt',
    'svr_mono_ecfp': 'RBF-SVR (aaECFP)',
    'rf_mono_ecfp': 'RF (aaECFP)',
    'xgboost_mono_ecfp': 'XGBoost (aaECFP)',
    'svr_pep_ecfp_counts': 'RBF-SVR (pepECFP)',
    'rf_pep_ecfp_counts': 'RF (pepECFP)',
    'xgboost_pep_ecfp_counts': 'XGBoost (pepECFP)',
    'chemeleon_graph': 'CheMeleon',
    # 'mlp_pepland_0130': 'Pepland'
}

# Optional: enforce a specific plotting order to match the list above
METHOD_ORDER = [METHOD_DISPLAY[m] for m in MODELDESC_KEEP]

METRICS = ['r2', 'spearman_corr', 'pearson_corr']
METRIC_DISPLAY = {'r2': '$R^{2}$', 'spearman_corr': 'Spearman', 'pearson_corr': 'Pearson'}
SPLIT_TYPES = ['cv', 'time', 'walkforward']

def run_anova(df_in, value_col):
    """One-way ANOVA across methods."""
    groups = [v[value_col].values for _, v in df_in.groupby("method")]
    groups = [g for g in groups if len(g) > 0 and np.isfinite(g).any()]
    if len(groups) < 2:
        return np.nan
    return f_oneway(*groups)[1]

def make_melt_df(df, metric, split_type):
    """
    Returns a dataframe pooled across all series with columns:
    - method: display label
    - value: metric value
    - split_type, fold: kept for context
    """
    sel = df[
        (df['model_desc'].isin(MODELDESC_KEEP)) &
        (df['split_type'] == split_type) &
        (df[metric].notna())
    ].copy()
    if sel.empty:
        return sel.assign(method=sel.get('model_desc', np.nan), value=sel.get(metric, np.nan))
    sel['method'] = sel['model_desc'].map(lambda x: METHOD_DISPLAY.get(x, x))
    sel['value'] = sel[metric]
    # Optional: order methods as categories for consistent plotting
    sel['method'] = pd.Categorical(sel['method'], categories=METHOD_ORDER, ordered=True)
    return sel[['method', 'value', 'split_type', 'fold']]

def plot_tukey_combined(df, metric, split_type, figsize=(7, 5), fig_idx=0):
    mdf = make_melt_df(df, metric, split_type)
    # temp solution:
    mdf["method"] = mdf['method'].str.replace("PepOpt", "ZZZPepOpt")

    if mdf.empty:
        print(f"No data for metric={metric}, split_type={split_type}")
        return None

    # Ensure at least two methods with data
    methods = mdf['method'].dropna().unique()
    if len(methods) < 2 or mdf['value'].isna().all():
        print(f"Insufficient data for Tukey: metric={metric}, split_type={split_type}")
        return None

    # Best method by mean (maximize the metric)
    best_method = mdf.groupby('method')['value'].mean().sort_values(ascending=False).index[0]

    # Tukey HSD
    tukey = pairwise_tukeyhsd(endog=mdf['value'], groups=mdf['method'].astype(str), alpha=0.05)

    # Plot
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    tukey.plot_simultaneous(comparison_name='ZZZPepOpt', ax=ax, figsize=figsize)

    pval = run_anova(mdf.rename(columns={'value': metric}), metric)
    metric_label = METRIC_DISPLAY.get(metric, metric)
    split_label = SPLIT_NAMES.get(split_type, split_type)
    # add (a), (b), etc. to figure title
    ax.set_title(f"({chr(97 + fig_idx)}) {split_label} ({metric_label})", fontsize=14, fontweight='bold')
    ax.set_xlabel(metric_label, fontsize=12, fontweight='bold')
    # set x-lim based on metric
    if metric in ['r2']:
        ax.set_xlim(-0.3, 0.9)
    elif metric in ['spearman_corr', 'pearson_corr']:
        ax.set_xlim(0.4, 0.95)

    # add grid 
    # ax.xaxis.grid(True, linestyle='--', color='lightgrey', alpha=0.7)
    # Improve readability: order y-axis labels to METHOD_ORDER if present
    try:
        # Get current y tick labels
        yticks = [t.get_text() for t in ax.get_yticklabels()]
        # Reorder if METHOD_ORDER is a superset of ticks
        order_idx = [yticks.index(m) for m in METHOD_ORDER if m in yticks]
        # Statsmodels doesn't expose a direct reordering; we leave label order as produced.
        # If you want strict order, we can switch to a custom interval-plot implementation.
    except Exception:
        pass

    # temp solution to get Pepopt first
    yticks = [t.get_text().replace('ZZZ', '').replace("ECFP", "") for t in ax.get_yticklabels()]
    # yticks = wrap_labels(yticks, width=12)
    # print(yticks)
    ax.set_yticklabels(yticks, rotation=30)

    fig.tight_layout()
    return fig

def plot_all_combined(combined_metrics, metrics=METRICS, split_types=SPLIT_TYPES):
    figs = {}
    i = 0
    for split_type in split_types:
        for metric in metrics:
            print(f"Plotting pooled across series: metric={metric}, split={split_type}...")
            fig = plot_tukey_combined(combined_metrics, metric, split_type, fig_idx=i, figsize=(5,4))
            figs[(split_type, metric)] = fig
            i += 1
    return figs

# Run plotting
figures = plot_all_combined(combined_metrics)

# Optionally save figures
if SAVE_FIGS:
    fig_idx = 0
    for (split_type, metric), fig in figures.items():
        if fig is not None:
            fname = f"{chr(97 + fig_idx)}-tukey_combined_{split_type}_{metric}.svg"
            fig.savefig(os.path.join(SAVE_FIGS_DIR, fname), format='svg', dpi=300)
            fig_idx += 1



In [ ]:
unique_vals = combined_metrics['model_desc'].unique()
print(unique_vals)


# Define model properties
# possible hatch patterns '///', '\\\\', 'xx', '++', '..'
model_properties = [
    {'method': 'bayesian_ridge_mono_ecfp',      'label': 'PepOpt',     'color': '#382E9B',   'hatch': ''},
    {'method': 'svr_mono_ecfp',     'label': 'RBF-SVR (aaECFP)',       'color': '#C20202',   'hatch': ''},
    {'method': 'rf_mono_ecfp',    'label': 'RF (aaECFP)',       'color': '#00857C',   'hatch': ''},
    {'method': 'xgboost_mono_ecfp','label': 'XGBoost (aaECFP)',  'color': '#FF7B00FF', 'hatch': ''},
    {'method': 'svr_pep_ecfp_counts',     'label': 'RBF-SVR (pepECFP)',       'color': '#C20202',   'hatch': '///'},    
    {'method': 'rf_pep_ecfp_counts',     'label': 'RF (pepECFP)',       'color': '#00857C',   'hatch': '///'},
    {'method': 'xgboost_pep_ecfp_counts','label': 'XGBoost (pepECFP)',  'color': '#FF7B00FF', 'hatch': '///'},
    {'method': 'chemeleon_graph',       'label': 'CheMeleon','color': '#0AC9C9B3', 'hatch': ''},
]

selected_models = [m['method'] for m in model_properties]
model_labels = [m['label'] for m in model_properties]
colors = [m['color'] for m in model_properties]
hatches = [m['hatch'] for m in model_properties]

metrics = ['r2', 'spearman_corr', 'pearson_corr']
metric_labels = {'r2': 'R2', 'spearman_corr': 'Spearman Corr.', 'pearson_corr': 'Pearson Corr.'}
split_types = ['cv', 'time', 'walkforward'] # , 'walkforward'
split_labels = {'walkforward': 'Walk Forward', 'cv': 'Random Split', 'time': 'Time Split'}


from matplotlib.patches import Patch
from matplotlib.lines import Line2D


# Filter the data
filtered_metrics = combined_metrics[combined_metrics['model_desc'].isin(selected_models)].copy()
# Use short x-tick labels (e.g., numbers)
short_xticks = [str(i+1) for i in range(len(selected_models))]

for split_type in split_types:
    split_data = filtered_metrics[filtered_metrics['split_type'] == split_type]
    
    # Prepare data for each metric
    fig, axes = plt.subplots(2, 3, figsize=(16, 5), 
                            gridspec_kw={'height_ratios': [10, 1]})  # 2nd row for legend
    
    for j, metric in enumerate(metrics):
        ax = axes[0, j]
        # Gather data for each model in the correct order
        data = [
            split_data[split_data['model_desc'] == m]['{}'.format(metric)].dropna().values
            for m in selected_models
        ]
        
        # Create boxplot with short x-tick labels
        bp = ax.boxplot(
            data,
            patch_artist=True,
            labels=short_xticks,
            medianprops=dict(color='black', linewidth=2),
            showfliers = False
        )
        
        # Set colors and hatches
        for patch, color, hatch in zip(bp['boxes'], colors, hatches):
            patch.set_facecolor(color)
            patch.set_hatch(hatch)
            patch.set_edgecolor('black')
            patch.set_linewidth(1.5)
        
        # Overlay mean points
        for idx, vals in enumerate(data):
            if len(vals) > 0:
                mean_val = np.mean(vals)
                ax.scatter(idx + 1, mean_val, color='red', s=100, marker='D',
                           zorder=10, edgecolors='black', linewidth=1, label='Mean' if idx == 0 else "")
        
        # ax.set_xlabel('Model', fontsize=12, fontweight='bold')
        ax.set_ylabel(metric_labels[metric], fontsize=12, fontweight='bold')
        # ax.tick_params(axis='x', labelsize=11)
        ax.set_xticks([])
        ax.tick_params(axis='y', labelsize=11)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('bold')
        ax.grid(axis='y', alpha=0.8, linestyle='--')
    
    # Create legend-only subplot
    ax_legend = axes[1, 1]  # Center subplot in the bottom row
    ax_legend.axis('off')
    legend_elements = [
        Patch(facecolor=color, edgecolor='black', hatch=hatch, label=f"{tick}: {label}")
        for color, hatch, tick, label in zip(colors, hatches, short_xticks, model_labels)
    ]
    mean_marker = Line2D([0], [0], marker='D', color='w', markerfacecolor='red',
                         markeredgecolor='black', markersize=8, linewidth=0, markeredgewidth=2, label='Mean')
    # legend_elements.append(mean_marker)
    # ax_legend.legend(handles=legend_elements, loc='center', ncol=2, fontsize=12, title="Model", title_fontsize=13, frameon=True)
    
    # Hide other legend axes
    axes[1, 0].axis('off')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# make an extra plot for legend
# Prepare legend elements
legend_elements = [
    Patch(facecolor=color, edgecolor='black', hatch=hatch, label=f"{tick}: {label}")
    for color, hatch, tick, label in zip(colors, hatches, short_xticks, model_labels)
]

# Create a new figure for the legend
fig_legend, ax_legend = plt.subplots(figsize=(6, 2))
ax_legend.axis('off')  # Hide the axis

# Add the legend to the figure
ax_legend.legend(
    handles=legend_elements,
    loc='center',
    ncol=4,  # Adjust columns as needed
    fontsize=12,
    title="Model",
    title_fontsize=13,
    frameon=True
)

plt.tight_layout()
plt.show()


In [ ]:
# Define model properties
# possible hatch patterns '///', '\\\\', 'xx', '++', '..'
model_properties = [
    {'method': 'bayesian_ridge_mono_ecfp',      'label': 'PepOpt',     'color': '#382E9B',   'hatch': ''},
    {'method': 'svr_mono_ecfp',     'label': 'RBF-SVR (monoECFP)',       'color': '#C20202',   'hatch': ''},
    {'method': 'rf_mono_ecfp',    'label': 'RF (monoECFP)',       'color': '#00857C',   'hatch': ''},
    {'method': 'xgboost_mono_ecfp','label': 'XGBoost (monoECFP)',  'color': '#FF7B00FF', 'hatch': ''},
    {'method': 'svr_pep_ecfp_counts',     'label': 'RBF-SVR (pepECFP-Counts)',       'color': '#C20202',   'hatch': '///'},    
    {'method': 'rf_pep_ecfp_counts',     'label': 'RF (pepECFP-Counts)',       'color': '#00857C',   'hatch': '///'},
    {'method': 'xgboost_pep_ecfp_counts','label': 'XGBoost (pepECFP-Counts)',  'color': '#FF7B00FF', 'hatch': '///'},
    {'method': 'chemeleon_graph',       'label': 'CheMeleon','color': '#0AC9C9B3', 'hatch': ''},
    {'method': 'mlp_pepland_0130',       'label': 'Pepland','color': '#800080', 'hatch': ''},
    {'method': 'mlp_pepland_orig_ckpt',       'label': 'Pepland org','color': '#800080', 'hatch': ''},
]

selected_models = [m['method'] for m in model_properties]
model_labels = [m['label'] for m in model_properties]
colors = [m['color'] for m in model_properties]
hatches = [m['hatch'] for m in model_properties]

metrics = ['r2', 'spearman_corr', 'pearson_corr']
metric_labels = {'r2': 'R2', 'spearman_corr': 'Spearman Corr.', 'pearson_corr': 'Pearson Corr.'}
split_types = ['cv', 'time', 'walkforward'] # , 'walkforward'
split_labels = {'walkforward': 'Walk Forward', 'cv': 'Random Split', 'time': 'Time Split'}

# Filter the data
filtered_metrics = combined_metrics[combined_metrics['model_desc'].isin(selected_models)].copy()

# create a pd, each row is a model_desc
# list of columns: model_desc r2_cv, spearman_cv, pearson_cv, r2_time, spearman_time, pearson_time, r2_walkforward, spearman_walkforward, pearson_walkforward
# for each metric column, store the mean \pm std
# Create summary DataFrame with mean ± std for each model and metric combination
summary_data = []

for model in selected_models:
    model_data = {'model_desc': model}
    
    # Get data for this specific model
    model_metrics = filtered_metrics[filtered_metrics['model_desc'] == model]
    # calculate average training and testing time for each model
    # model_data['avg_train_time'] = model_metrics['train_time'].mean()
    # model_data['avg_test_time'] = model_metrics['test_time'].mean()
    # Calculate mean ± std for each metric and split type combination
    for split_type in split_types:
        split_data = model_metrics[model_metrics['split_type'] == split_type]
        
        for metric in metrics:
            if not split_data.empty and metric in split_data.columns:
                mean_val = split_data[metric].mean()
                std_val = split_data[metric].std()
                
                # Format as "mean ± std" with appropriate precision
                if pd.notna(mean_val) and pd.notna(std_val):
                    formatted_value = f"{mean_val:.2f} ± {std_val:.2f}"
                else:
                    formatted_value = "N/A"
            else:
                formatted_value = "N/A"
            
            # Create column name like 'r2_cv', 'spearman_time', etc.
            column_name = f"{metric}_{split_type}"
            model_data[column_name] = formatted_value
    summary_data.append(model_data)

# Create the summary DataFrame
summary_df = pd.DataFrame(summary_data)

# Reorder columns for better readability
column_order = ['model_desc']
for split_type in split_types:
    for metric in metrics:
        column_order.append(f"{metric}_{split_type}")

summary_df = summary_df[column_order]

# Display the summary
print("Model Performance Summary (Mean ± Std):")
print("=" * 80)
print(summary_df.to_string(index=False))

# # Optional: Save to CSV
summary_df.to_csv(f'{output_dir}/model_performance_summary.csv', index=False)
summary_df.to_excel(f'{output_dir}/model_performance_summary.xlsx', index=False)


### Fig 4 - series-wise performance


In [ ]:
## 
def plot_metrics_in_subplots(combined_metrics, metric_cols, metric_labels, metric_titles, 
                            included_methods, method_names, method_colors, split_type, n_cols=2):
    """
    Plot metrics in subplots with configurable number of columns
    
    Parameters:
    -----------
    combined_metrics : DataFrame
        DataFrame containing the metrics data
    metric_cols : list
        List of metric column names
    metric_labels : dict
        Dictionary mapping metric names to display labels
    metric_titles : dict
        Dictionary mapping metric names to plot titles
    included_methods : list
        List of methods to include in the plots
    method_names : dict
        Dictionary mapping method names to display names
    method_colors : dict
        Dictionary mapping method names to colors
    split_type : str
        Type of split (e.g., 'cv', 'test')
    n_cols : int, optional
        Number of columns in the subplot grid (default: 2)
    """
    # Calculate mean and std for each method and fold
    combined_metrics = combined_metrics[combined_metrics['split_type'] == split_type]
    stats = combined_metrics.groupby(['model_desc', 'series'])[metric_cols].agg(['mean', 'std']).reset_index()
    # Calculate number of rows needed based on number of metrics and columns
    n_metrics = len(metric_cols)
    n_rows = math.ceil(n_metrics / n_cols)
    
    # Create figure with subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 5*n_rows))
    
    # Flatten axes array for easier indexing if there are multiple subplots
    if n_metrics > 1:
        axes = axes.flatten()
    else:
        axes = [axes]  # Make it iterable for single subplot case
    
    # Plot each metric in its own subplot
    for i, metric in enumerate(metric_cols):
        ax = axes[i]
        
        # Plot for each method
        for method in included_methods:
            # Filter data for this method
            method_data = stats[stats['model_desc'] == method]
            
            # Extract mean and std for this metric
            means = method_data[(metric, 'mean')].values
            stds = method_data[(metric, 'std')].values
            series = method_data['series'].values
            
            # Plot mean line with specific color
            x_pos = np.arange(len(series))
            display_name = method_names.get(method, method)  # Use original if not in mapping
            color = method_colors[method]  # Get the color for this method
            
            ax.plot(x_pos, means, marker='o', label=display_name, color=color)
            
            # Add shaded region for standard deviation with matching color
            ax.fill_between(x_pos, means - stds, means + stds, alpha=0.2, color=color)
        
        # Add labels and legend
        ax.set_xlabel('Series name', fontweight='bold')
        ax.set_ylabel(metric_labels[metric], fontweight='bold')
        # ax.set_title(metric_titles[metric])
        ax.grid(True, linestyle='--', alpha=0.7)
        # ax.legend(title='Method')
        
        # Adjust x-axis ticks
        ax.set_xticks(x_pos)
        ax.set_xticklabels(series, rotation=30)
        # 
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('bold')
    
    # Hide any unused subplots
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()
    
    # make an extra figure for legend
    handles, labels = axes[0].get_legend_handles_labels()
    # Create a new figure for the legend
    legend_fig = plt.figure(figsize=(4, 2))
    plt.figlegend(handles, labels, loc='center', ncol=1, title='Method')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


    return fig, legend_fig

# Define series names:
SERIES_NAMES = {
    'Target_A_B-8154': 'A-1',
    'Target_A_B-8154 Macrolactam': 'A-2',
    'Target_A_B-8209': 'A-3',
    'Target_B_4R-5594': 'B-1',
    'Target_B_4R-2390': 'B-2',
    'Target_B_4R-9882': 'B-3',
    'Target_C_L-300_Macrolactam': 'C-1',
    'Target_C_L-320_Macrolactam': 'C-2',
    'Target_C_L-878': 'C-3',
    'Target_D_X-link 2-11': 'D-1',
    'Target_E_B-3503': 'E-1'
}

# Define model properties
# possible hatch patterns '///', '\\\\', 'xx', '++', '..'
model_properties = [
    {'method': 'bayesian_ridge_mono_ecfp',      'label': 'PepOpt',     'color': '#382E9B',   'hatch': ''},
    {'method': 'svr_mono_ecfp',     'label': 'RBF-SVR (monoECFP)',       'color': '#C20202',   'hatch': ''},
    {'method': 'rf_mono_ecfp',    'label': 'RF (monoECFP)',       'color': '#00857C',   'hatch': ''},
    {'method': 'xgboost_mono_ecfp','label': 'XGBoost (monoECFP)',  'color': '#FF7B00FF', 'hatch': ''},
    {'method': 'chemeleon_graph',       'label': 'CheMeleon','color': '#0AC9C9B3', 'hatch': ''},
]

selected_models = [m['method'] for m in model_properties]
model_labels = [m['label'] for m in model_properties]
colors = [m['color'] for m in model_properties]
hatches = [m['hatch'] for m in model_properties]

metrics = ['r2', 'spearman_corr', 'pearson_corr']
metric_labels = {'r2': 'R2', 'spearman_corr': 'Spearman Corr.', 'pearson_corr': 'Pearson Corr.'}
split_types = ['cv', 'time'] # , 'walkforward'
split_labels = {'walkforward': 'Walk Forward', 'cv': 'Random Split', 'time': 'Time Split'}

# Define better titles for plots
metric_titles = metric_labels

# Define color mapping for methods
method_colors = dict(zip(selected_models, colors))
method_names = dict(zip(selected_models, model_labels))
combined_metrics['series'] = combined_metrics['series'].replace('B-8154 Macrolactam', 'B-8154 Macro.')
for split_type in split_types:
    # Call the function with 2 columns (you can change this parameter)
    _, _ = plot_metrics_in_subplots(
        combined_metrics=combined_metrics,
        metric_cols=metrics,
        metric_labels=metric_labels,
        metric_titles=metric_titles,
        included_methods=selected_models,
        method_names=method_names,
        method_colors=method_colors,
        split_type=split_type,
        n_cols=3  # You can change this to control the number of columns
    )


In [ ]:
def plot_metrics_separately(combined_metrics, metric_cols, metric_labels, metric_titles, 
                           included_methods, method_names, method_colors, split_types,
                           series_names_mapping):
    """
    Plot each metric-split type combination in a separate figure
    
    Parameters:
    -----------
    combined_metrics : DataFrame
        DataFrame containing the metrics data
    metric_cols : list
        List of metric column names
    metric_labels : dict
        Dictionary mapping metric names to display labels
    metric_titles : dict
        Dictionary mapping metric names to plot titles
    included_methods : list
        List of methods to include in the plots
    method_names : dict
        Dictionary mapping method names to display names
    method_colors : dict
        Dictionary mapping method names to colors
    split_types : list
        List of split types (e.g., ['cv', 'test'])
    series_names_mapping : dict
        Dictionary mapping original series names to display names
    """
    
    figures = []
    legend_fig = None
    
    # Plot each metric-split combination separately
    for split_type in split_types:
        # Filter data for this split type
        split_data = combined_metrics[combined_metrics['split_type'] == split_type].copy()
        
        # Apply series name mapping
        split_data['series_display'] = split_data['series'].map(series_names_mapping)
        
        # Calculate mean and std for each method and fold
        stats = split_data.groupby(['model_desc', 'series_display'])[metric_cols].agg(['mean', 'std']).reset_index()
        
        # Sort by series display name for consistent ordering
        unique_series = sorted(stats['series_display'].unique())
        
        for metric in metric_cols:
            # Create a single figure for this metric-split combination
            fig, ax = plt.subplots(1, 1, figsize=(10, 4))
            
            # Plot for each method
            for method in included_methods:
                # Filter data for this method
                method_data = stats[stats['model_desc'] == method].copy()
                
                # Sort by series display name
                method_data = method_data.sort_values('series_display')
                
                # Extract mean and std for this metric
                means = method_data[(metric, 'mean')].values
                stds = method_data[(metric, 'std')].values
                series = method_data['series_display'].values
                
                # Plot mean line with specific color
                x_pos = np.arange(len(series))
                display_name = method_names.get(method, method)
                color = method_colors[method]
                
                ax.plot(x_pos, means, marker='o', label=display_name, color=color, linewidth=2, markersize=8)
                
                # Add shaded region for standard deviation
                ax.fill_between(x_pos, means - stds, means + stds, alpha=0.2, color=color)
            
            # Add labels and formatting
            ax.set_xlabel('Series Name', fontweight='bold', fontsize=16)
            ax.set_ylabel(metric_labels[metric], fontweight='bold', fontsize=16)
            # set ticks font size
            ax.tick_params(axis='x', labelsize=16)
            ax.tick_params(axis='y', labelsize=16)
            # Add title indicating split type and metric
            split_label = {'cv': 'Random Split', 'time': 'Time Split', 'walkforward': 'Walk Forward'}.get(split_type, split_type)
            ax.set_title(f'{metric_titles[metric]} - {split_label}', fontweight='bold', fontsize=14)
            
            ax.grid(True, linestyle='--', alpha=0.7)
            
            # Adjust x-axis ticks
            ax.set_xticks(x_pos)
            ax.set_xticklabels(series, rotation=0, ha='center')
            
            # Make tick labels bold
            for label in ax.get_xticklabels() + ax.get_yticklabels():
                label.set_fontweight('bold')
            
            if split_type == split_types[0]:
                ax.legend(fontsize=16, frameon=True, ncol=2, loc='lower left')

            plt.tight_layout()
            figures.append((fig, f'{metric}_{split_type}'))
            plt.show()

            if SAVE_FIGS:
                fig_filename = f'series_performance_{metric}_{split_type}.svg'
                fig.savefig(os.path.join(SAVE_FIGS_DIR, fig_filename), format='svg', dpi=300)
                print(f"Saved figure to {os.path.join(SAVE_FIGS_DIR, fig_filename)}")
    
    # Create a single legend figure (only once)
    if figures:  # Check if we have at least one figure
        # Get handles and labels from the first plot
        handles, labels = figures[0][0].axes[0].get_legend_handles_labels()
        
        # Create a new figure for the legend
        legend_fig = plt.figure(figsize=(10, 4))
        plt.figlegend(handles, labels, loc='center', ncol=5, fontsize=16, frameon=True,
              columnspacing=1.0,      # Horizontal space between columns
              handletextpad=0.5,      # Space between marker and label text
              handlelength=1.5,       # Length of the legend marker (default is 2.0)
              labelspacing=0.3)       # Vertical space between entries (default is 0.5)

        # reduce the gaps between legend entries
        plt.subplots_adjust(wspace=0.1, hspace=0.1)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

        if SAVE_FIGS:
            legend_fig_filename = f'series_performance_legend.svg'
            legend_fig.savefig(os.path.join(SAVE_FIGS_DIR, legend_fig_filename), format='svg', dpi=300)
            print(f"Saved legend to {os.path.join(SAVE_FIGS_DIR, legend_fig_filename)}")
    
    return figures, legend_fig


# Define series names
SERIES_NAMES = {
    'Target_A_B-8154': 'A-1',
    'Target_A_B-8154 Macrolactam': 'A-2',
    'Target_A_B-8209': 'A-3',
    'Target_B_4R-5594': 'B-1',
    'Target_B_4R-2390': 'B-2',
    'Target_B_4R-9882': 'B-3',
    'Target_C_L-300_Macrolactam': 'C-1',
    'Target_C_L-320_Macrolactam': 'C-2',
    'Target_C_L-878': 'C-3',
    'Target_D_X-link 2-11': 'D-1',
    'Target_E_B-3503': 'E-1'
}

# Define model properties
model_properties = [
    {'method': 'bayesian_ridge_mono_ecfp', 'label': 'PepOpt', 'color': '#382E9B', 'hatch': ''},
    {'method': 'svr_mono_ecfp', 'label': 'RBF-SVR (aaECFP)', 'color': '#C20202', 'hatch': ''},
    {'method': 'rf_mono_ecfp', 'label': 'RF (aaECFP)', 'color': '#00857C', 'hatch': ''},
    {'method': 'xgboost_mono_ecfp', 'label': 'XGBoost (aaECFP)', 'color': '#FF7B00FF', 'hatch': ''},
    {'method': 'chemeleon_graph', 'label': 'CheMeleon', 'color': '#0AC9C9B3', 'hatch': ''},
]

selected_models = [m['method'] for m in model_properties]
model_labels = [m['label'] for m in model_properties]
colors = [m['color'] for m in model_properties]
hatches = [m['hatch'] for m in model_properties]

metrics = ['r2', 'spearman_corr', 'pearson_corr']
metric_labels = {'r2': '$R^2$', 'spearman_corr': 'Spearman Corr.', 'pearson_corr': 'Pearson Corr.'}
split_types = ['cv', 'time', 'walkforward']  # , 'walkforward'
split_labels = {'walkforward': 'Walk Forward', 'cv': 'Random Split', 'time': 'Time Split'}

# Define better titles for plots
metric_titles = metric_labels

# Define color mapping for methods
method_colors = dict(zip(selected_models, colors))
method_names = dict(zip(selected_models, model_labels))

# Call the revised function
figures, legend_fig = plot_metrics_separately(
    combined_metrics=combined_metrics,
    metric_cols=metrics,
    metric_labels=metric_labels,
    metric_titles=metric_titles,
    included_methods=selected_models,
    method_names=method_names,
    method_colors=method_colors,
    split_types=split_types,
    series_names_mapping=SERIES_NAMES
)
